In [1]:
#Rodas apenas uma vez
#!pip install -U pip setuptools wheel
#!pip install -U spacy
#!python -m spacy download pt_core_news_sm

In [27]:
!python --user pip install spacy==3.2.0

unknown option --user
usage: python [option] ... [-c cmd | -m mod | file | -] [arg] ...
Try `python -h' for more information.


In [2]:
import pandas as pd
import spacy

In [3]:
dados_treino = pd.read_csv("dados/treinop2.csv")
dados_treino.sample(5)

,title,text,date,category,subcategory,link
27278,Promotoria pede 18 meses de prisão para pai de...,A Promotoria da Espanha pediu nesta terça-feir...,2015-06-10,esporte,NaN,http://www1.folha.uol.com.br/esporte/2015/10/1...
8854,Governo Alckmin reduz repasse e dá calote de R...,A gestão Geraldo Alckmin (PSDB) deu um calote ...,2016-02-03,cotidiano,NaN,http://www1.folha.uol.com.br/cotidiano/2016/03...
85000,Alalaô: Salgueiro e Vila Isabel se destacam em...,PEDRO SOARES DO RIO Na primeira noite de desfi...,2015-02-16,cotidiano,NaN,http://www1.folha.uol.com.br/cotidiano/2015/02...
48830,O Dia Olímpico e a Lava Jato,Embora haja motivo para uma festa um pouco mai...,2015-06-23,colunas,edgardalves,http://www1.folha.uol.com.br/colunas/edgardalv...
48836,ANS não cobra planos por atendimentos feitos n...,O governo tem deixado de receber uma quantia b...,2015-08-02,cotidiano,NaN,http://www1.folha.uol.com.br/cotidiano/2015/02...


In [4]:
nlp = spacy.load("pt_core_news_sm")

In [5]:
texto = "Rio de Janeiro é uma cidade maravilhosa"
doc = nlp(texto)

In [6]:
doc

Rio de Janeiro é uma cidade maravilhosa

In [7]:
type(doc[2])

spacy.tokens.token.Token

In [8]:
textos_para_tratamento = (titulos.lower() for titulos in dados_treino["title"])

In [9]:
def trata_textos(doc):
    tokens_validos = []
    for token in doc:
        e_valido = not token.is_stop and token.is_alpha
        if e_valido:
            tokens_validos.append(token.text)

    if len(tokens_validos) > 2:
        return  " ".join(tokens_validos)

texto = "Rio de Janeiro 1231231 ***** @#$ é uma cidade maravilhosa!"
doc = nlp(texto)
trata_textos(doc)

'Rio Janeiro cidade maravilhosa'

In [10]:
texto = "Rio de Janeiro 1231231 ***** @#$ é uma cidade maravilhosa!"
doc = nlp(texto)
trata_textos(doc)

'Rio Janeiro cidade maravilhosa'

In [11]:
from time import time

t0 = time()
textos_tratados = [trata_textos(doc) for doc in nlp.pipe(textos_para_tratamento,
                                                        batch_size = 1000,
                                                        n_process = -1)]

tf = time() - t0

print(tf/60)

1.8973514636357625


In [12]:
titulos_tratados = pd.DataFrame({"titulo": textos_tratados})
titulos_tratados.head()

,titulo
0,polêmica marine le pen abomina negacionistas h...
1,macron le pen turno frança revés siglas tradic...
2,apesar larga vitória legislativas macron terá ...
3,governo antecipa balanço alckmin anuncia queda...
4,queda maio atividade econômica sobe junho bc


In [13]:
from gensim.models import Word2Vec

w2v_modelo = Word2Vec(sg = 0,
                      window = 2,
                      min_count = 5,
                      alpha = 0.03,
                      min_alpha = 0.007)

In [14]:
w2v_modelo

In [15]:
print(len(titulos_tratados))
titulos_tratados = titulos_tratados.dropna().drop_duplicates()
print(len(titulos_tratados))

90000
84466


In [16]:
lista_lista_tokens = [titulo.split(" ") for titulo in titulos_tratados.titulo]

In [17]:
import logging

logging.basicConfig(format="%(asctime)s : - %(message)s", level = logging.INFO)

w2v_modelo = Word2Vec(sg = 0,
                      window = 2,
                      size = 300,
                      min_count = 5,
                      alpha = 0.03,
                      min_alpha = 0.007)

w2v_modelo.build_vocab(lista_lista_tokens, progress_per=5000)

TypeError: __init__() got an unexpected keyword argument 'size'

In [ ]:
dir(w2v_modelo)

In [ ]:
w2v_modelo.corpus_count

In [ ]:
w2v_modelo.train(lista_lista_tokens, 
                 total_examples=w2v_modelo.corpus_count,
                 epochs = 30)

In [ ]:
w2v_modelo.wv.most_similar("google")

In [ ]:
w2v_modelo.wv.most_similar("microsoft")

In [ ]:
w2v_modelo.wv.most_similar("barcelona")

In [ ]:
w2v_modelo.wv.most_similar("messi")

In [ ]:
w2v_modelo.wv.most_similar("gm")

In [ ]:
#Treinamento do modelo Skip-Gram
w2v_modelo_sg = Word2Vec(sg = 1,
                      window = 5,                      
                      min_count = 5,
                      alpha = 0.03,
                      min_alpha = 0.007)

w2v_modelo_sg.build_vocab(lista_lista_tokens, progress_per=5000)

w2v_modelo_sg.train(lista_lista_tokens, 
                 total_examples=w2v_modelo_sg.corpus_count,
                 epochs = 30)

In [ ]:
w2v_modelo_sg.wv.most_similar("google")

In [ ]:
w2v_modelo.wv.most_similar("google")

In [ ]:
w2v_modelo_sg.wv.most_similar("gm")

In [ ]:
w2v_modelo.wv.most_similar("gm")

In [ ]:
w2v_modelo.wv.save_word2vec_format("modelo_cbow.txt", binary=False)
w2v_modelo_sg.wv.save_word2vec_format("modelo_skipgram.txt", binary=False)